# Starbucks Walk-Distance Cannibalization: 171 Store Isochrones via OSMnx

*How much do Starbucks stores cannibalize each other's walking catchments in Manhattan?*

**Series context:** This is Theme 2C (walk-distance analysis) in the Starbucks data science series. See also:
- [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) — EDA & competitor mapping (Theme 0)
- [Starbucks 10-K NLP](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) — Keyword trends, LDA topics (Theme 1)
- [Starbucks Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) — Moran's I, LISA, Ripley's K (Theme 2A)
- [Starbucks Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) — Demand-supply scoring & LFS (Theme 2B)
- [Data Pipeline](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) — How the dataset was built
- [US Expansion Choropleth](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) — 30-year state-level animation
- [LDA Topic Explorer](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) — Interactive LDA exploration (Theme 1F)

### What this notebook does

Notebook B used **straight-line distances** (Euclidean) to measure competition and demand. But pedestrians walk along streets, not as-the-crow-flies. This notebook uses **OSMnx** to compute actual walking distances on Manhattan's street network:

1. Download Manhattan's full walking network (~12K nodes, ~38K edges)
2. Compute 5-minute walk isochrones (400m) for all 171 Starbucks stores
3. Measure **catchment overlap** between Starbucks pairs → cannibalization index
4. Count **competitors within walk-distance** vs. straight-line distance
5. Compare **walk-distance to subway** vs. straight-line distance
6. Map **coverage gaps** — Manhattan areas >5 min walk from any Starbucks

**Dataset:** [Manhattan Cafe Wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)

## Section 0 — Setup & Data Loading

In [ ]:
!pip install -q geopandas folium plotly osmnx networkx shapely

import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union
from shapely.strtree import STRtree
from pathlib import Path
from IPython.display import display, HTML
import warnings, time
warnings.filterwarnings("ignore")

# ----- Data path auto-detect (Kaggle vs local) -----
KAGGLE_PATHS = [
    Path("/kaggle/input/manhattan-cafe-wars"),
    Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
]
LOCAL_DIR = Path("../../data/processed")

DATA_DIR = None
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    for p in KAGGLE_PATHS:
        if p.exists():
            DATA_DIR = p
            break
if DATA_DIR is None:
    DATA_DIR = LOCAL_DIR

print(f"Data directory: {DATA_DIR}")
print(f"ON_KAGGLE: {ON_KAGGLE}")

def show_map(m, height=500):
    """Render folium map in Kaggle notebook."""
    from IPython.display import IFrame
    import tempfile, os
    tmp = tempfile.NamedTemporaryFile(suffix=".html", delete=False)
    m.save(tmp.name)
    return IFrame(tmp.name, width="100%", height=height)

In [ ]:
# ----- Load data -----
starbucks = pd.read_csv(DATA_DIR / "manhattan_starbucks_osm.csv")
cafes = pd.read_csv(DATA_DIR / "manhattan_cafes_osm.csv")
mta = pd.read_csv(DATA_DIR / "manhattan_mta_ridership_summary.csv")
enriched = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")

print(f"Starbucks stores: {len(starbucks)}")
print(f"All cafés:        {len(cafes):,}")
print(f"MTA stations:     {len(mta)}")
print(f"Enriched cols:    {len(enriched.columns)}")

# Sanity checks
assert starbucks["lat"].between(40.6, 40.9).all(), "Latitude out of Manhattan range"
assert starbucks["lon"].between(-74.1, -73.9).all(), "Longitude out of Manhattan range"
assert len(starbucks) > 150, f"Expected ~171 stores, got {len(starbucks)}"
print("\n✓ All sanity checks passed.")

## Section 1 — Manhattan Walking Network

We download the full Manhattan walking network from OpenStreetMap via OSMnx. This graph represents every sidewalk, crosswalk, and pedestrian path as a network of nodes (intersections) and edges (street segments with real-world lengths in meters).

In [ ]:
# ----- Download full Manhattan walking network -----
print("Downloading Manhattan walking network (this may take 30-60 seconds)...")
t0 = time.time()
G = ox.graph_from_place("Manhattan, New York, USA", network_type="walk")
elapsed = time.time() - t0

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
print(f"  Download time: {elapsed:.1f}s")
print(f"  Nodes: {n_nodes:,}")
print(f"  Edges: {n_edges:,}")

# Compute basic graph stats
edge_lengths = [d["length"] for _, _, d in G.edges(data=True)]
print(f"  Edge length — median: {np.median(edge_lengths):.0f}m, "
      f"mean: {np.mean(edge_lengths):.0f}m, max: {max(edge_lengths):.0f}m")
print(f"  Total network length: {sum(edge_lengths)/1000:.0f} km")

In [ ]:
# ----- Fig. 1: Manhattan Walking Network -----
fig1, ax1 = ox.plot_graph(G, figsize=(8, 14), node_size=0, edge_linewidth=0.3,
                          edge_color="#444444", bgcolor="white", show=False, close=False)

# Overlay Starbucks locations
ax1.scatter(starbucks["lon"], starbucks["lat"], c="#00704A", s=12, 
            zorder=5, alpha=0.8, label=f"Starbucks ({len(starbucks)})")
ax1.legend(loc="lower right", fontsize=10)
ax1.set_title("Fig. 1 — Manhattan Walking Network with Starbucks Locations", fontsize=13, pad=10)
plt.tight_layout()
plt.show()

## Section 2 — Computing All 171 Isochrones

An **isochrone** is the area reachable within a given travel time from a point. We compute 5-minute walking isochrones (400m at 80 m/min) for every Starbucks in Manhattan.

For each store:
1. Find the nearest node on the walking graph
2. Use Dijkstra's algorithm to find all nodes within 400m network distance
3. Build a convex hull polygon from the reachable nodes

We also compute 10-minute (800m) isochrones for comparison.

In [ ]:
# ----- Compute isochrones for all 171 stores -----
from shapely.geometry import MultiPoint

def compute_isochrone(G, lat, lon, max_dist_m=400):
    """Compute walk-distance isochrone polygon from a point.
    Returns convex hull of all reachable nodes within max_dist_m."""
    center_node = ox.nearest_nodes(G, lon, lat)
    subgraph = nx.ego_graph(G, center_node, radius=max_dist_m, distance="length")
    reachable_nodes = list(subgraph.nodes)
    node_coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in reachable_nodes]
    if len(node_coords) < 3:
        return None
    return MultiPoint(node_coords).convex_hull

WALK_DIST_5MIN = 400   # meters (5 min at 80 m/min)
WALK_DIST_10MIN = 800  # meters (10 min)

print(f"Computing isochrones for {len(starbucks)} stores...")
t0 = time.time()

iso_5min = []
iso_10min = []
for idx, row in starbucks.iterrows():
    iso5 = compute_isochrone(G, row["lat"], row["lon"], WALK_DIST_5MIN)
    iso10 = compute_isochrone(G, row["lat"], row["lon"], WALK_DIST_10MIN)
    iso_5min.append(iso5)
    iso_10min.append(iso10)

elapsed = time.time() - t0
print(f"  Completed in {elapsed:.1f}s")

# Add to DataFrame
starbucks["iso_5min"] = iso_5min
starbucks["iso_10min"] = iso_10min

valid_5 = starbucks["iso_5min"].notna().sum()
valid_10 = starbucks["iso_10min"].notna().sum()
print(f"  Valid 5-min isochrones: {valid_5}/{len(starbucks)}")
print(f"  Valid 10-min isochrones: {valid_10}/{len(starbucks)}")

# Area statistics
areas_5 = starbucks["iso_5min"].dropna().apply(lambda p: p.area * 111_320 * 85_000)  # approx m²
areas_10 = starbucks["iso_10min"].dropna().apply(lambda p: p.area * 111_320 * 85_000)
print(f"\n5-min isochrone area — median: {areas_5.median():,.0f} m², "
      f"mean: {areas_5.mean():,.0f} m², range: {areas_5.min():,.0f}–{areas_5.max():,.0f} m²")
print(f"10-min isochrone area — median: {areas_10.median():,.0f} m², "
      f"mean: {areas_10.mean():,.0f} m²")

In [ ]:
# ----- Fig. 2: Sample isochrones — Midtown cluster -----
# Show ~15 stores in the Midtown area (lat 40.748-40.762, lon -73.990--73.974)
midtown_mask = (
    starbucks["lat"].between(40.748, 40.762) &
    starbucks["lon"].between(-73.990, -73.974) &
    starbucks["iso_5min"].notna()
)
midtown = starbucks[midtown_mask]
print(f"Midtown stores shown: {len(midtown)}")

center = [midtown["lat"].mean(), midtown["lon"].mean()]
m2 = folium.Map(location=center, zoom_start=15, tiles="CartoDB positron")

colors = px.colors.qualitative.Set3
for i, (_, row) in enumerate(midtown.iterrows()):
    color = colors[i % len(colors)]
    folium.GeoJson(
        row["iso_5min"].__geo_interface__,
        style_function=lambda x, c=color: {
            "fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.2,
        },
        tooltip=f"{row['name']}: 5-min walk",
    ).add_to(m2)
    folium.CircleMarker(
        [row["lat"], row["lon"]], radius=5, color="#00704A",
        fill=True, fill_opacity=0.9, popup=row["name"],
    ).add_to(m2)

m2.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
    "font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
    "border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
    f"Fig. 2 — 5-Minute Walk Isochrones in Midtown ({len(midtown)} stores)</div>"
))
show_map(m2)

## Section 3 — Catchment Overlap & Cannibalization Index

For each pair of Starbucks stores, we compute the **Intersection over Union (IoU)** of their 5-minute walk isochrones:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

A high IoU means two stores are competing for the same walking population — a measure of **cannibalization**.

In [ ]:
# ----- Compute pairwise overlap using spatial index -----
valid_stores = starbucks[starbucks["iso_5min"].notna()].copy().reset_index(drop=True)
n = len(valid_stores)
print(f"Computing pairwise overlap for {n} stores...")
print(f"  Total pairs: {n*(n-1)//2:,}")

t0 = time.time()

# Build spatial index for fast intersection queries
polys = valid_stores["iso_5min"].tolist()
tree = STRtree(polys)

# Find all intersecting pairs
overlap_pairs = []
for i in range(n):
    candidates = tree.query(polys[i])
    for j in candidates:
        if j <= i:
            continue
        if polys[i].intersects(polys[j]):
            inter_area = polys[i].intersection(polys[j]).area
            union_area = polys[i].union(polys[j]).area
            iou = inter_area / union_area if union_area > 0 else 0
            if iou > 0.01:  # Skip negligible overlaps
                overlap_pairs.append({
                    "store_i": i, "store_j": j,
                    "name_i": valid_stores.loc[i, "name"],
                    "name_j": valid_stores.loc[j, "name"],
                    "iou": iou,
                    "inter_area_pct_i": inter_area / polys[i].area * 100,
                    "inter_area_pct_j": inter_area / polys[j].area * 100,
                })

elapsed = time.time() - t0
overlap_df = pd.DataFrame(overlap_pairs)
print(f"  Completed in {elapsed:.1f}s")
print(f"  Overlapping pairs (IoU > 1%): {len(overlap_df):,}")
print(f"  Stores with at least 1 overlap: {len(set(overlap_df['store_i']) | set(overlap_df['store_j']))}")

if len(overlap_df) > 0:
    print(f"\nIoU statistics:")
    print(f"  Mean:   {overlap_df['iou'].mean():.3f}")
    print(f"  Median: {overlap_df['iou'].median():.3f}")
    print(f"  Max:    {overlap_df['iou'].max():.3f}")
    print(f"  Pairs with IoU > 0.3: {(overlap_df['iou'] > 0.3).sum()}")
    print(f"  Pairs with IoU > 0.5: {(overlap_df['iou'] > 0.5).sum()}")

In [ ]:
# ----- Fig. 3: Distribution of overlap IoU -----
fig3 = px.histogram(overlap_df, x="iou", nbins=40,
                    title="Fig. 3 — Distribution of Pairwise Catchment Overlap (IoU)",
                    labels={"iou": "Intersection over Union (IoU)", "count": "Number of Pairs"},
                    color_discrete_sequence=["#00704A"])
fig3.add_vline(x=overlap_df["iou"].median(), line_dash="dash", line_color="red",
               annotation_text=f"Median IoU = {overlap_df['iou'].median():.3f}")
fig3.update_layout(template="plotly_white", height=400)
fig3.show()

In [ ]:
# ----- Fig. 4: Top 20 most cannibalized pairs on map -----
top20 = overlap_df.nlargest(20, "iou")
print("Top 10 most overlapping Starbucks pairs (highest cannibalization):")
for _, row in top20.head(10).iterrows():
    print(f"  IoU={row['iou']:.3f}  {row['name_i']}  ↔  {row['name_j']}")

# Map the top pairs
all_lats = [valid_stores.loc[int(r["store_i"]), "lat"] for _, r in top20.iterrows()] + \
           [valid_stores.loc[int(r["store_j"]), "lat"] for _, r in top20.iterrows()]
all_lons = [valid_stores.loc[int(r["store_i"]), "lon"] for _, r in top20.iterrows()] + \
           [valid_stores.loc[int(r["store_j"]), "lon"] for _, r in top20.iterrows()]
center = [np.mean(all_lats), np.mean(all_lons)]

m4 = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
for _, row in top20.iterrows():
    si, sj = int(row["store_i"]), int(row["store_j"])
    ri, rj = valid_stores.loc[si], valid_stores.loc[sj]
    # Draw line between pair
    folium.PolyLine(
        [[ri["lat"], ri["lon"]], [rj["lat"], rj["lon"]]],
        color="red", weight=2 + row["iou"] * 5, opacity=0.6,
        tooltip=f"IoU={row['iou']:.3f}",
    ).add_to(m4)

# Add all Starbucks as small dots
for _, row in valid_stores.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]], radius=3, color="#00704A",
        fill=True, fill_opacity=0.7,
    ).add_to(m4)

m4.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
    "font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
    "border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
    "Fig. 4 — Top 20 Most Cannibalized Starbucks Pairs (red = high IoU)</div>"
))
show_map(m4)

## Section 4 — Competitors Within Walking Catchment

Notebook B counted competitors within straight-line radius (250m, 500m). Here we count competitors **actually reachable within a 5-minute walk** using the isochrone polygons. This is more realistic because Manhattan's grid means straight-line distances often underestimate or overestimate walking distances.

In [ ]:
# ----- Count competitors within walk-distance isochrones -----
competitors = cafes[cafes["brand_category"] != "starbucks"].copy()
comp_points = [Point(lon, lat) for lon, lat in zip(competitors["lon"], competitors["lat"])]
comp_tree = STRtree(comp_points)

walk_comp_counts = []
for _, row in valid_stores.iterrows():
    iso = row["iso_5min"]
    if iso is None:
        walk_comp_counts.append(0)
        continue
    hits = comp_tree.query(iso)
    count = sum(1 for h in hits if iso.contains(comp_points[h]))
    walk_comp_counts.append(count)

valid_stores["competitors_walk_5min"] = walk_comp_counts

# Merge straight-line competitor counts from enriched data
straight_cols = ["osm_id", "n_starbucks_500m", "n_dunkin_500m", "n_other_cafe_500m"]
valid_stores = valid_stores.merge(
    enriched[straight_cols], on="osm_id", how="left", suffixes=("", "_enriched")
)
valid_stores["competitors_straight_500m"] = (
    valid_stores["n_dunkin_500m"].fillna(0) + valid_stores["n_other_cafe_500m"].fillna(0)
).astype(int)

print("Competitors within 5-min walk vs 500m straight-line:")
print(f"  Walk-distance — mean: {valid_stores['competitors_walk_5min'].mean():.1f}, "
      f"median: {valid_stores['competitors_walk_5min'].median():.0f}, "
      f"max: {valid_stores['competitors_walk_5min'].max()}")
print(f"  Straight 500m — mean: {valid_stores['competitors_straight_500m'].mean():.1f}, "
      f"median: {valid_stores['competitors_straight_500m'].median():.0f}, "
      f"max: {valid_stores['competitors_straight_500m'].max()}")

corr = valid_stores[["competitors_walk_5min", "competitors_straight_500m"]].corr().iloc[0, 1]
print(f"\n  Correlation: r = {corr:.3f}")

In [ ]:
# ----- Fig. 5: Walk-distance vs straight-line competitor count -----
fig5 = px.scatter(
    valid_stores, x="competitors_straight_500m", y="competitors_walk_5min",
    title="Fig. 5 — Walk-Distance vs Straight-Line Competitor Counts",
    labels={"competitors_straight_500m": "Competitors within 500m (straight-line)",
            "competitors_walk_5min": "Competitors within 5-min walk (network)"},
    color_discrete_sequence=["#00704A"],
    hover_data=["name"],
)
# Add y=x reference line
max_val = max(valid_stores["competitors_straight_500m"].max(),
              valid_stores["competitors_walk_5min"].max())
fig5.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode="lines",
                          line=dict(dash="dash", color="gray"), name="y = x"))
fig5.update_layout(template="plotly_white", height=500,
                   annotations=[dict(x=0.95, y=0.05, xref="paper", yref="paper",
                                     text=f"r = {corr:.3f}", showarrow=False,
                                     font=dict(size=14))])
fig5.show()

# Where do they differ most?
valid_stores["comp_diff"] = valid_stores["competitors_walk_5min"] - valid_stores["competitors_straight_500m"]
overest = (valid_stores["comp_diff"] < -2).sum()
underest = (valid_stores["comp_diff"] > 2).sum()
print(f"\nStores where straight-line OVERESTIMATES competitors (>2 difference): {overest}")
print(f"Stores where straight-line UNDERESTIMATES competitors (>2 difference): {underest}")

## Section 5 — Walk-Distance to Subway vs Straight-Line

Notebook B used Euclidean distance (`mta_dist_m`) to assign each Starbucks to its nearest subway station. Here we compute the **actual walking distance** along streets and compare.

In [ ]:
# ----- Compute walk-distance to nearest MTA station -----
print("Computing walk-distance to nearest MTA station for each store...")
t0 = time.time()

walk_dists_to_mta = []
nearest_station_walk = []

# Pre-compute nearest graph nodes for MTA stations
mta_nodes = [ox.nearest_nodes(G, row["lon"], row["lat"]) for _, row in mta.iterrows()]

for _, store in valid_stores.iterrows():
    store_node = ox.nearest_nodes(G, store["lon"], store["lat"])
    min_dist = float("inf")
    min_station = None
    for j, mta_node in enumerate(mta_nodes):
        try:
            dist = nx.shortest_path_length(G, store_node, mta_node, weight="length")
            if dist < min_dist:
                min_dist = dist
                min_station = mta.iloc[j]["station_name"]
        except nx.NetworkXNoPath:
            continue
    walk_dists_to_mta.append(min_dist if min_dist < float("inf") else np.nan)
    nearest_station_walk.append(min_station)

valid_stores["mta_walk_dist_m"] = walk_dists_to_mta
valid_stores["mta_station_walk"] = nearest_station_walk
elapsed = time.time() - t0
print(f"  Completed in {elapsed:.1f}s")

# Merge straight-line distance
if "mta_dist_m" not in valid_stores.columns:
    valid_stores = valid_stores.merge(
        enriched[["osm_id", "mta_dist_m"]], on="osm_id", how="left"
    )

valid_mta = valid_stores.dropna(subset=["mta_walk_dist_m", "mta_dist_m"])
print(f"\nWalk-distance to MTA — median: {valid_mta['mta_walk_dist_m'].median():.0f}m, "
      f"mean: {valid_mta['mta_walk_dist_m'].mean():.0f}m")
print(f"Straight-line to MTA — median: {valid_mta['mta_dist_m'].median():.0f}m, "
      f"mean: {valid_mta['mta_dist_m'].mean():.0f}m")
print(f"Walk/straight ratio — median: {(valid_mta['mta_walk_dist_m'] / valid_mta['mta_dist_m']).median():.2f}x")

In [ ]:
# ----- Fig. 6: Walk-distance vs straight-line to MTA -----
fig6 = px.scatter(
    valid_mta, x="mta_dist_m", y="mta_walk_dist_m",
    title="Fig. 6 — Walk-Distance vs Straight-Line Distance to Nearest Subway Station",
    labels={"mta_dist_m": "Straight-line distance (m)",
            "mta_walk_dist_m": "Walk-distance on street network (m)"},
    color_discrete_sequence=["#00704A"],
    hover_data=["name"],
)
max_val = max(valid_mta["mta_dist_m"].max(), valid_mta["mta_walk_dist_m"].max())
fig6.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode="lines",
                          line=dict(dash="dash", color="gray"), name="y = x (same distance)"))

corr_mta = valid_mta[["mta_dist_m", "mta_walk_dist_m"]].corr().iloc[0, 1]
fig6.update_layout(template="plotly_white", height=500,
                   annotations=[dict(x=0.95, y=0.05, xref="paper", yref="paper",
                                     text=f"r = {corr_mta:.3f}", showarrow=False,
                                     font=dict(size=14))])
fig6.show()

print(f"Correlation between walk and straight-line distance: r = {corr_mta:.3f}")
print(f"Walk distance is typically {(valid_mta['mta_walk_dist_m'] / valid_mta['mta_dist_m']).median():.1f}x "
      f"the straight-line distance (Manhattan grid effect)")

## Section 6 — Coverage Gaps: Where Is Manhattan Underserved?

By merging all 171 isochrones, we can identify Manhattan areas that are **more than a 5-minute walk from any Starbucks**. Overlaying these gaps with the demand data from Notebook B reveals potential expansion opportunities.

In [ ]:
# ----- Compute coverage union -----
valid_isos = [p for p in valid_stores["iso_5min"] if p is not None]
print(f"Merging {len(valid_isos)} isochrone polygons...")

t0 = time.time()
coverage_5min = unary_union(valid_isos)
elapsed = time.time() - t0
print(f"  Union computed in {elapsed:.1f}s")

# Load Manhattan boundary for gap analysis
tracts = gpd.read_file(DATA_DIR / "manhattan_tracts_lisa.geojson")
manhattan_boundary = tracts.unary_union

# Compute gap area
gap = manhattan_boundary.difference(coverage_5min)

coverage_area = coverage_5min.area * 111_320 * 85_000  # approx m²
manhattan_area = manhattan_boundary.area * 111_320 * 85_000
gap_area = gap.area * 111_320 * 85_000

print(f"\nManhattan total area:     {manhattan_area/1e6:.1f} km²")
print(f"5-min walk coverage:      {coverage_area/1e6:.1f} km² ({coverage_area/manhattan_area*100:.1f}%)")
print(f"Coverage gap (>5 min):    {gap_area/1e6:.1f} km² ({gap_area/manhattan_area*100:.1f}%)")

In [ ]:
# ----- Fig. 7: Coverage map -----
m7 = folium.Map(
    location=[40.758, -73.985], zoom_start=12, tiles="CartoDB positron"
)

# Coverage area (green)
folium.GeoJson(
    coverage_5min.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "#00704A", "color": "#00704A",
        "weight": 0.5, "fillOpacity": 0.15,
    },
    name="5-min walk coverage",
).add_to(m7)

# Gap area (red)
if not gap.is_empty:
    folium.GeoJson(
        gap.__geo_interface__,
        style_function=lambda x: {
            "fillColor": "#FF4444", "color": "#FF4444",
            "weight": 0.5, "fillOpacity": 0.3,
        },
        name="Coverage gap (>5 min walk)",
    ).add_to(m7)

# Store locations
for _, row in valid_stores.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]], radius=2.5, color="#00704A",
        fill=True, fill_opacity=0.8,
    ).add_to(m7)

folium.LayerControl().add_to(m7)
m7.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
    "font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
    "border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
    f"Fig. 7 — Starbucks 5-Min Walk Coverage (green) vs Gaps (red)</div>"
))
show_map(m7, height=600)

In [ ]:
# ----- Fig. 8: Per-store cannibalization score -----
# For each store, compute what fraction of its 5-min isochrone overlaps with OTHER Starbucks isochrones
cannibal_scores = []
for i in range(len(valid_stores)):
    my_iso = valid_stores.iloc[i]["iso_5min"]
    if my_iso is None:
        cannibal_scores.append(0)
        continue
    others = [valid_stores.iloc[j]["iso_5min"] for j in range(len(valid_stores))
              if j != i and valid_stores.iloc[j]["iso_5min"] is not None]
    others_union = unary_union(others)
    overlap_area = my_iso.intersection(others_union).area
    cannibal_scores.append(overlap_area / my_iso.area * 100 if my_iso.area > 0 else 0)

valid_stores["cannibalization_pct"] = cannibal_scores

print("Per-store cannibalization (% of catchment overlapping with other Starbucks):")
print(f"  Mean:   {valid_stores['cannibalization_pct'].mean():.1f}%")
print(f"  Median: {valid_stores['cannibalization_pct'].median():.1f}%")
print(f"  Max:    {valid_stores['cannibalization_pct'].max():.1f}%")
print(f"  Stores with >50% cannibalization: {(valid_stores['cannibalization_pct'] > 50).sum()}")
print(f"  Stores with >75% cannibalization: {(valid_stores['cannibalization_pct'] > 75).sum()}")

fig8 = px.histogram(valid_stores, x="cannibalization_pct", nbins=30,
                    title="Fig. 8 — Per-Store Cannibalization Score (% of catchment overlapping)",
                    labels={"cannibalization_pct": "Cannibalization (%)"},
                    color_discrete_sequence=["#00704A"])
fig8.update_layout(template="plotly_white", height=400)
fig8.show()

## Section 7 — Summary & Limitations

### Key Findings

| Metric | Value |
|--------|-------|
| Valid isochrones computed | 171 stores |
| Walk/straight-line ratio to subway | ~1.3-1.4x (Manhattan grid effect) |
| Walk-distance competitor correlation with straight-line | High but imperfect |
| Manhattan coverage by 5-min walk to Starbucks | See Section 6 |

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| **Convex hull approximation** | Isochrones may overestimate reachable area near rivers/parks | Could use alpha shapes or buffer-based isochrones |
| **Uniform walk speed (80 m/min)** | Doesn't account for hills, traffic lights, crosswalks | Use time-based routing with elevation data |
| **Static OSM network** | Road closures, construction not reflected | Accept as snapshot limitation |
| **No indoor connections** | Subway-connected buildings, underground passages ignored | Would require custom graph modifications |
| **Point-of-entry approximation** | Store matched to nearest graph node, not actual entrance | Typically <20m error, negligible |

### Series Navigation

| Notebook | Theme | Link |
|----------|-------|------|
| Manhattan Cafe Wars | Theme 0: EDA & competitor mapping | [Open](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| Starbucks 10-K NLP | Theme 1: keyword trends, LDA topics | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| Starbucks Spatial Clustering | Theme 2A: Moran's I, LISA, Ripley's K | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| Starbucks Location Fitness | Theme 2B: demand-supply scoring & LFS | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| **Walk-Distance Analysis** | **Theme 2C: this notebook** | — |
| Data Pipeline | Pipeline: EDGAR & OSM to CSV | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| US Expansion Choropleth | 30-year state-level animation | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| 1F | [LDA Topic Explorer](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) | Interactive LDA, word clouds, CEO-era analysis |

**Datasets:** [Manhattan Cafe Wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars) · [Starbucks 30-Year NLP Corpus](https://www.kaggle.com/datasets/shiratoriseto/starbucks-30year-nlp-corpus)